In [5]:
# ==== B1 · 环境与常量 ====
# 所有块都依赖这里：读连接、SQL 入口、字段常量，以及统一的示例参数。
from pathlib import Path
import pandas as pd
import quant
from quant import schema
from quant.db import get_conn, query

DB_PATH = Path("data/databases/quant.duckdb")  # 相对项目根目录；换库只改这一行
START, END = "2024-01-01", "2024-12-31"  # 统一示例区间
STOCK = "SH600000"  # 个股示例：浦发银行
INDEX = "000300.SH"  # 指数示例：沪深300
SW_INDEX = "801010.SI"  # 申万一级行业示例：农林牧渔

pd.set_option("display.max_columns", 60)  # 日线有 43 列，别被 pandas 折叠掉
pd.set_option("display.width", 200)

quant.__all__


['board_of',
 'infer_exchange',
 'normalize_code',
 'load_calendar',
 'load_financial',
 'load_index_constituents',
 'load_index_basic',
 'load_index_daily',
 'load_index_weight',
 'load_label',
 'load_label_wide',
 'load_members',
 'load_panel',
 'load_panel_wide',
 'load_stock_basic',
 'load_suspend_segments',
 'load_sw_index_daily']

In [7]:
# ==== B2 · 库总览：表清单与行数 ====
# md 是行情与标签，ref 是参照表，evt 是事件明细，meta 是版本/口径/异常。
# 表名要写全 ``schema.table``，SQL 里不带 schema 会找不到表。
tables = query(
    "SELECT schema_name AS schema, table_name AS tbl FROM duckdb_tables() ORDER BY 1, 2"
)
tables["rows"] = [
    query(f"SELECT count(*) AS n FROM {s}.{t}").n[0]
    for s, t in zip(tables.schema, tables.tbl)
]
tables

,schema,tbl,rows
0,evt,suspend_daily,234916
1,md,calendar,6480
2,md,daily_bar,17955297
3,md,daily_hfq,17955297
4,md,daily_qfq,17955297
5,md,fwd_return,17955297
6,md,index_daily,280086
7,md,suspend_segment,71319
8,md,sw_index_daily,104777
9,meta,anomaly,57157


In [19]:
# ==== B3 · 字段单位口径：meta.data_dictionary ====
# 不知道某列的单位就看这里；库里的字段口径以这张表为准。
dictionary = query(
    """
    SELECT table_name, column_name, unit, description
    FROM meta.data_dictionary
    WHERE table_name IN ('daily_bar', 'daily_hfq', 'daily_qfq', 'fwd_return')
    ORDER BY table_name, column_name
    """
)
dictionary

,table_name,column_name,unit,description
0,daily_bar,amount_cny,元,成交额，Tushare amount × 1000
1,daily_bar,can_buy_*/can_sell_*,布尔,保守口径的可交易性
2,daily_bar,change,元,官方涨跌额
3,daily_bar,factor,无量纲,后复权因子，后复权价 = 原始价 × factor
4,daily_bar,is_delisting,布尔,当日是否处于退市整理期
5,daily_bar,is_st,布尔,当日是否 ST，来源 namechange
6,daily_bar,limit_up/limit_down,元,涨跌停价，优先官方 stk_limit，缺失时规则推算
7,daily_bar,open/high/low/close,元,未复权四价
8,daily_bar,pre_close,元,前收盘价 = close - change
9,daily_bar,ret,小数,官方日收益 = change / pre_close


In [9]:
# ==== B4 · 清洗异常明细：meta.anomaly ====
# 四类异常：收益口径差异、有因子跳变无除权、涨跌停价越界、有除权无因子跳变。
# 注意 trade_date 在这张表里是字符串，跨表 join 要先 CAST 成 DATE。
anomaly = query(
    """
    SELECT kind, count(*) AS n_rows,
           min(trade_date) AS first_day, max(trade_date) AS last_day
    FROM meta.anomaly GROUP BY 1 ORDER BY 2 DESC
    """
)
anomaly

,kind,n_rows,first_day,last_day
0,return_mismatch,28715,2000-01-05,2026-09-16
1,factor_jump_without_ex_div,26099,2003-07-24,2026-09-16
2,limit_out_of_range,2193,2000-01-26,2022-03-30
3,ex_div_without_factor_jump,150,2001-08-09,2026-08-21


In [11]:
# ==== B5 · 直连 SQL：get_conn / query ====
# 读取层没封装的东西（聚合、窗口、自定义 join）直接用 SQL；连接默认只读，
# 可以和其它进程并存。写库要显式传 read_only=False，正式库请统一用建库脚本重建。
with get_conn() as conn:  # 等价于 get_conn(DB_PATH, read_only=True)
    manifest = conn.execute("SELECT data_version, generated_at FROM meta.manifest").df()
    n_calendar = conn.execute("SELECT count(*) AS n FROM md.calendar").fetchone()[0] # type: ignore

# query() 每次新开一个只读连接；参数用 ? 占位，别做字符串拼接
recent = query(
    """
    SELECT stock_code, trade_date, close
    FROM md.daily_hfq
    WHERE stock_code = ? AND trade_date >= ?
    ORDER BY trade_date DESC LIMIT 5
    """,
    [STOCK, "2026-09-01"],
)
recent

,stock_code,trade_date,close
0,SH600000,2026-09-28,159.176984
1,SH600000,2026-09-24,156.396600
2,SH600000,2026-09-23,156.049052
3,SH600000,2026-09-22,157.091696
4,SH600000,2026-09-21,156.570374


In [12]:
# ==== B6 · 代码归一：normalize_code / infer_exchange / board_of ====
# 库里的股票代码统一是 SH600000 / SZ000001 / BJ920000，其它写法的原始数据
# 都要先过 normalize_code；指数代码例外，保留 Tushare 写法 000300.SH。
raw_codes = ["600000.SH", "sh.600000", "000001", "T600018.SH", "920000", "688001.SH"]
code_map = pd.DataFrame({"原始写法": raw_codes})
code_map["归一结果"] = [quant.normalize_code(code) for code in raw_codes]
code_map["交易所"] = [quant.infer_exchange(code[-6:]) for code in code_map["归一结果"]]
code_map["板块"] = [quant.board_of(code[-6:]) for code in code_map["归一结果"]]
code_map

,原始写法,归一结果,交易所,板块
0,600000.SH,SH600000,SH,main_sh
1,sh.600000,SH600000,SH,main_sh
2,000001,SZ000001,SZ,main_sz
3,T600018.SH,SH600018,SH,main_sh
4,920000,BJ920000,BJ,bse
5,688001.SH,SH688001,SH,star


In [13]:
# ==== B7 · 可用字段清单：quant.schema ====
# load_panel 的 fields 只能从这些列里选，选错会报「未知字段」。
# 口径组：ADJUST_MODES = ('raw', 'hfq', 'qfq')，对应三张同构日线表 DAILY_TABLES；
# 标签组：LABEL_HORIZONS = (1, 3, 5, 10, 15, 20)，LABEL_PRICES = ('close', 'open')。
field_groups = {
    "价格": schema.PRICE_COLUMNS,
    "量额": schema.MARKET_COLUMNS,
    "复权与收益": schema.ADJUST_COLUMNS,
    "生命周期与停牌": schema.LIFECYCLE_COLUMNS,
    "估值": schema.VALUATION_COLUMNS,
    "涨跌停": schema.LIMIT_COLUMNS,
    "可交易性": schema.TRADEABLE_COLUMNS,
    "审计来源": schema.AUDIT_COLUMNS,
}
fields = pd.DataFrame(
    [{"分组": group, "字段": name} for group, cols in field_groups.items() for name in cols]
)
fields

,分组,字段
0,价格,open
1,价格,high
2,价格,low
3,价格,close
4,量额,volume_share
5,量额,amount_cny
6,量额,vwap
7,复权与收益,factor
8,复权与收益,change
9,复权与收益,pre_close


In [20]:
# ==== B8 · 交易日历：load_calendar(start, end) ====
# 所有对齐都要靠它：行情只覆盖每个交易日，日历用来生成完整交易日轴。
cal = quant.load_calendar(start=START, end=END)
cal

,trade_date,is_open
0,2024-01-02,1
1,2024-01-03,1
2,2024-01-04,1
3,2024-01-05,1
4,2024-01-08,1
...,...,...
237,2024-12-25,1
238,2024-12-26,1
239,2024-12-27,1
240,2024-12-30,1


In [22]:
# ==== B9 · 股票基础表：load_stock_basic(status) ====
# status 取 "listed" / "delisted"，也接受 Tushare 的 "L" / "D" / "P"，不传就是全部；
# 非法值会直接抛 ValueError，不会悄悄返回全表。单行明细看 basic.head()。
basic = quant.load_stock_basic()
listed = quant.load_stock_basic(status="listed")
delisted = quant.load_stock_basic(status="delisted")
basic.groupby(["list_status", "board"]).size().rename("只数").reset_index()

,list_status,board,只数
0,D,bse,5
1,D,chinext,41
2,D,main_sh,143
3,D,main_sz,146
4,D,star,3
5,L,bse,347
6,L,chinext,1407
7,L,main_sh,1702
8,L,main_sz,1494
9,L,other,1


In [24]:
# ==== B10 · 停牌区间：load_suspend_segments() ====
# 全表 71,319 段；is_long 是清洗阶段的长停牌标记。日线表的 status == 'suspended'
# 只能看单日，要整段剔除就用这张表。
seg = quant.load_suspend_segments()
seg.sort_values("start_date", ascending=False).head(8)

,stock_code,start_date,end_date,n_days,is_long
69356,SZ300211,2026-09-28,2026-09-28,1,False
68125,SZ300096,2026-09-24,2026-09-24,1,False
66721,SZ002860,2026-09-22,2026-09-28,4,False
71070,SZ300585,2026-09-21,2026-09-23,3,False
67948,SZ300082,2026-09-21,2026-09-28,5,False
33332,SH603400,2026-09-15,2026-09-28,9,False
31724,SH601059,2026-09-15,2026-09-28,9,False
32887,SH601995,2026-09-15,2026-09-22,6,False


In [26]:
# ==== B11 · 日线长表：load_panel(...) ====
# 主力接口。fields 不传返回全部 43 列；adjust 取 raw / hfq / qfq；
# drop_suspended=True 剔掉停牌行。不做全市场研究时不要省略 start / end，
# 否则会把 1795 万行整表读进内存。
panel = quant.load_panel(
    start=START,
    end="2024-01-31",
    stocks=[STOCK],
    fields=["open", "high", "low", "close", "ret", "limit_down", "is_limit_up",
            "status", "can_buy_close"],
    adjust = "hfq",
    drop_suspended=True
)
panel.head()

,stock_code,trade_date,open,high,low,close,ret,limit_down,is_limit_up,status,can_buy_close
0,SH600000,2024-01-02,102.99705,103.30775,102.53100,102.53100,-0.003021,5.96,False,trading,True
1,SH600000,2024-01-03,102.37565,103.30775,102.37565,103.15240,0.006061,5.94,False,trading,True
2,SH600000,2024-01-04,103.15240,103.61845,101.75425,102.84170,-0.003012,5.98,False,trading,True
3,SH600000,2024-01-05,102.53100,105.01660,102.37565,103.77380,0.009063,5.96,False,trading,True
4,SH600000,2024-01-08,103.77380,104.23985,101.90960,102.37565,-0.013473,6.01,False,trading,True


In [27]:
# ==== B12 · 三种复权口径对比：raw / hfq / qfq ====
# 三张表结构完全一致，只有四价与 vwap 不同：
#   hfq = raw × factor；qfq = raw × factor / factor_T，factor_T 是该股最后一个交易日因子。
# 所以同一只股票在不同时间取 qfq，数值会变；做回测统一用 hfq。
raw = quant.load_panel(start=START, end="2024-01-10", stocks=[STOCK],
                       fields=["close"], adjust="raw")
hfq = quant.load_panel(start=START, end="2024-01-10", stocks=[STOCK],
                       fields=["close", "factor"], adjust="hfq")
qfq = quant.load_panel(start=START, end="2024-01-10", stocks=[STOCK],
                       fields=["close"], adjust="qfq")
compare = (
    raw.rename(columns={"close": "close_raw"})
    .merge(hfq.rename(columns={"close": "close_hfq"}), on=["stock_code", "trade_date"])
    .merge(qfq.rename(columns={"close": "close_qfq"}), on=["stock_code", "trade_date"])
)
compare["hfq/raw"] = (compare.close_hfq / compare.close_raw).round(6)
compare.head()

,stock_code,trade_date,close_raw,close_hfq,factor,close_qfq,hfq/raw
0,SH600000,2024-01-02,6.60,102.53100,15.535,5.900250,15.535
1,SH600000,2024-01-03,6.64,103.15240,15.535,5.936009,15.535
2,SH600000,2024-01-04,6.62,102.84170,15.535,5.918129,15.535
3,SH600000,2024-01-05,6.68,103.77380,15.535,5.971768,15.535
4,SH600000,2024-01-08,6.59,102.37565,15.535,5.891310,15.535


In [31]:
# ==== B13 · 日线宽表：load_panel_wide(field, ...) ====
# 透视成「交易日 × 股票」，横截面运算（排序、标准化、分组）在宽表上做最省事。
close_wide = quant.load_panel_wide("close", start=START, end=END, adjust="hfq")
close_wide.head(3)

stock_code,BJ920000,BJ920001,BJ920002,BJ920006,BJ920008,BJ920010,BJ920014,BJ920016,BJ920017,BJ920019,BJ920021,BJ920023,BJ920026,BJ920030,BJ920033,BJ920039,BJ920046,BJ920047,BJ920057,BJ920058,BJ920060,BJ920061,BJ920062,BJ920066,BJ920075,BJ920077,BJ920082,BJ920087,BJ920088,BJ920089,...,SZ301567,SZ301568,SZ301571,SZ301577,SZ301578,SZ301580,SZ301585,SZ301586,SZ301587,SZ301588,SZ301589,SZ301591,SZ301592,SZ301596,SZ301598,SZ301600,SZ301603,SZ301606,SZ301607,SZ301608,SZ301611,SZ301613,SZ301617,SZ301618,SZ301622,SZ301626,SZ301628,SZ301631,SZ301633,SZ302132
trade_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2024-01-02,14.85234,16.869762,NaN,21.60207,NaN,23.478224,23.647552,NaN,17.634444,NaN,60.132825,8.406992,44.90,11.366608,97.205710,13.454616,11.006651,7883.17524,5.44,43.500894,NaN,33.591075,11.327184,NaN,22.508494,28.317543,NaN,14.264457,NaN,20.302560,...,NaN,44.23,NaN,NaN,73.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,264.79656
2024-01-03,14.57982,15.850161,NaN,20.44539,NaN,24.202390,23.141930,NaN,17.925477,NaN,62.449500,8.448304,48.71,11.881045,96.015436,13.440483,11.197000,7731.40894,5.49,42.915087,NaN,31.416525,11.026842,NaN,22.175446,28.814342,NaN,14.217573,NaN,20.181186,...,NaN,43.20,NaN,NaN,73.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.79264
2024-01-04,14.64795,16.159131,NaN,21.20580,NaN,24.202390,24.172621,NaN,17.666781,NaN,60.938625,8.345024,46.75,11.660572,99.126854,13.157823,10.704332,7753.08984,5.49,46.392135,NaN,32.046000,10.941030,NaN,22.189323,28.488853,NaN,14.510598,NaN,19.960506,...,NaN,43.35,NaN,NaN,75.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,257.74488


In [ ]:
# ==== B14 · 指数目录：load_index_basic(...) ====
# market 取 SSE / SZSE / CSI / SW，category 如「规模指数」「一级行业」。
# 目录有 10,609 行，含大量分级/货币等变体代码；只有 md.index_daily 里那 58 个
# 才真有日线，查代码时用 has_daily 过滤一下最稳。
live_codes = query("SELECT DISTINCT index_code FROM md.index_daily").index_code
index_catalog = quant.load_index_basic()
index_catalog["has_daily"] = index_catalog.index_code.isin(live_codes)
index_catalog[index_catalog.has_daily]

In [33]:
# ==== B15 · 指数日线：load_index_daily(index_code, ...) ====
# md.index_daily 共 58 个指数：30 宽基 + 10 风格 + 8 策略 + 10 中证全指行业。
# 399925.SZ（基本面50，到 2019-01-25）与 399984.SZ（300等权，到 2015-04-29）
# 已经停编，只有历史行情，别拿它们当基准。
idx_daily = quant.load_index_daily(INDEX, start=START, end=END)
idx_daily.head()

,index_code,index_name,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,000300.SH,沪深300,2024-01-02,3426.2684,3426.2684,3386.3522,3386.3522,3431.1099,-44.7577,-1.3045,116180726.0,1.840961e+08
1,000300.SH,沪深300,2024-01-03,3379.7501,3392.8504,3362.6617,3378.2971,3386.3522,-8.0551,-0.2379,105613851.0,1.650990e+08
2,000300.SH,沪深300,2024-01-04,3375.0207,3375.0207,3323.8647,3347.0519,3378.2971,-31.2452,-0.9249,106721013.0,1.655123e+08
3,000300.SH,沪深300,2024-01-05,3341.3061,3372.5444,3313.5542,3329.1114,3347.0519,-17.9405,-0.5360,126400574.0,1.822517e+08
4,000300.SH,沪深300,2024-01-08,3322.6051,3333.5160,3283.3421,3286.0564,3329.1114,-43.0550,-1.2933,117429485.0,1.707616e+08


In [ ]:
# ==== B16 · 申万行业日线：load_sw_index_daily(index_code, ...) ====
# 31 个申万一级行业，比 index_daily 多出行业整体 pe（市盈率） / pb（市净率） 与市值列；
# 覆盖从 2010-04-13 起，比中证系指数短。
sw_daily = quant.load_sw_index_daily(SW_INDEX, start=START, end=END)
sw_daily.head()

,index_code,index_name,trade_date,open,low,high,close,change,pct_change,vol,amount,pe,pb,float_mv,total_mv
0,801010.SI,农林牧渔,2024-01-02,2862.28,2850.55,2891.51,2876.23,14.00,0.49,118576.0,989268.0,50.98,2.70,60500800.0,133782000.0
1,801010.SI,农林牧渔,2024-01-03,2869.67,2839.65,2892.20,2851.25,-24.98,-0.87,99776.0,848990.0,50.54,2.68,59975300.0,132887000.0
2,801010.SI,农林牧渔,2024-01-04,2848.11,2824.23,2848.11,2830.11,-21.14,-0.74,77468.0,663095.0,50.15,2.66,59517700.0,131779000.0
3,801010.SI,农林牧渔,2024-01-05,2826.10,2785.53,2827.80,2795.17,-34.94,-1.23,87414.0,747269.0,49.53,2.62,58782900.0,130070000.0
4,801010.SI,农林牧渔,2024-01-08,2790.61,2723.63,2793.95,2723.63,-71.54,-2.56,96151.0,852816.0,48.42,2.56,57464800.0,126716000.0


In [35]:
# ==== B17 · 指数成分权重：load_index_weight(index_code, start, end) ====
# 12 个宽基的月末快照。con_code 是 Tushare 原写法（600050.SH），
# stock_code 是归一后的项目口径（SH600050），join 日线表必须用 stock_code。
# 旧名别名 load_index_constituents 只接 index_code，没有 start / end 参数，
# 其余行为与 load_index_weight 一致（下面 assert 了一下）。
snapshot = quant.load_index_weight(INDEX, start="2024-01-31", end="2024-01-31")
alias_full = quant.load_index_constituents(INDEX)
assert alias_full.equals(quant.load_index_weight(INDEX))
snapshot.head()

,index_code,con_code,weight,index_name,trade_date,stock_code
0,000300.SH,000001.SZ,0.564,沪深300,2024-01-31,SZ000001
1,000300.SH,000002.SZ,0.401,沪深300,2024-01-31,SZ000002
2,000300.SH,000063.SZ,0.430,沪深300,2024-01-31,SZ000063
3,000300.SH,000069.SZ,0.089,沪深300,2024-01-31,SZ000069
4,000300.SH,000100.SZ,0.467,沪深300,2024-01-31,SZ000100


In [37]:
# ==== B18 · 标签长表：load_label(start, end, horizons, price, stocks) ====
# 列名 fwd_ret_close_N / fwd_ret_open_N；price="close" 指 t 日收盘买入、t+N 收盘卖出，
# price="open" 指 t+1 开盘买入、t+1+N 开盘卖出，两者持有交易日数一致。
# 停牌段用上一有效交易日价格填平，停牌行与尾部不足 N 的行留空。
label = quant.load_label(start=START, end="2024-01-31", horizons=[1, 5, 20], price="close")
label.head(10)

,stock_code,trade_date,fwd_ret_close_1,fwd_ret_close_5,fwd_ret_close_20
0,BJ920000,2024-01-02,-0.018349,-0.148624,-0.264220
1,BJ920001,2024-01-02,-0.060440,-0.232601,-0.154457
2,BJ920006,2024-01-02,-0.053545,-0.111552,-0.372831
3,BJ920010,2024-01-02,0.030844,-0.159091,-0.292208
4,BJ920014,2024-01-02,-0.021382,-0.068257,-0.305099
5,BJ920017,2024-01-02,0.016504,-0.149144,-0.250611
6,BJ920021,2024-01-02,0.038526,-0.162479,-0.268007
7,BJ920023,2024-01-02,0.004914,-0.120393,-0.262899
8,BJ920026,2024-01-02,0.084855,-0.120490,-0.324722
9,BJ920030,2024-01-02,0.045259,-0.109914,-0.280172


In [38]:
# ==== B19 · 标签宽表：load_label_wide(horizon, ...) ====
# 单档标签的「交易日 × 股票」形态，和 load_panel_wide 对齐后可直接算 IC。
fwd5_wide = quant.load_label_wide(5, start=START, end=END, price="close")
fwd5_wide.head(3)

stock_code,BJ920000,BJ920001,BJ920002,BJ920006,BJ920008,BJ920010,BJ920014,BJ920016,BJ920017,BJ920019,BJ920021,BJ920023,BJ920026,BJ920030,BJ920033,BJ920039,BJ920046,BJ920047,BJ920057,BJ920058,BJ920060,BJ920061,BJ920062,BJ920066,BJ920075,BJ920077,BJ920082,BJ920087,BJ920088,BJ920089,...,SZ301567,SZ301568,SZ301571,SZ301577,SZ301578,SZ301580,SZ301585,SZ301586,SZ301587,SZ301588,SZ301589,SZ301591,SZ301592,SZ301596,SZ301598,SZ301600,SZ301603,SZ301606,SZ301607,SZ301608,SZ301611,SZ301613,SZ301617,SZ301618,SZ301622,SZ301626,SZ301628,SZ301631,SZ301633,SZ302132
trade_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2024-01-02,-0.148624,-0.232601,NaN,-0.111552,NaN,-0.159091,-0.068257,NaN,-0.149144,NaN,-0.162479,-0.120393,-0.120490,-0.109914,-0.118367,-0.170168,-0.148525,-0.090209,-0.125000,0.046916,NaN,-0.210903,-0.188131,NaN,-0.201603,-0.114943,NaN,-0.161873,NaN,-0.128804,...,NaN,-0.047253,NaN,NaN,-0.054795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.099075
2024-01-03,-0.127103,-0.170890,NaN,-0.031954,NaN,-0.168504,-0.042857,NaN,-0.143115,NaN,-0.188710,-0.080685,-0.194827,-0.129897,-0.117007,-0.161935,-0.130000,-0.042064,-0.112933,0.132100,NaN,-0.122769,-0.150454,NaN,-0.170839,-0.120690,NaN,-0.138500,NaN,-0.100055,...,NaN,-0.025694,NaN,NaN,-0.074353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.091201
2024-01-04,-0.131163,-0.148502,NaN,-0.045455,NaN,-0.168504,-0.096541,NaN,-0.111043,NaN,-0.161983,-0.086634,-0.170909,-0.111345,-0.127870,-0.128894,-0.079498,0.013423,-0.112933,0.047658,NaN,-0.080000,-0.137255,NaN,-0.178236,-0.051714,NaN,-0.161551,NaN,-0.097291,...,NaN,-0.007151,NaN,NaN,-0.046358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.059819


In [ ]:
# ==== B20 · 财务数据与行业成分：load_financial / load_financial_panel ====
# 财务原始记录按报告期读取；日面板按公告日次一交易日对齐，避免前视。
financial = quant.load_financial(
    STOCK, table="fina_indicator", start="2023-01-01", end="2024-12-31",
    fields=["report_date", "ann_date", "available_date", "roe_yearly", "netprofit_yoy"],
)
financial_panel = quant.load_financial_panel(
    "fina_indicator", start="2024-01-01", end="2024-12-31",
    fields=["roe_yearly", "netprofit_yoy"], stocks=[STOCK],
)
industry = quant.load_members("industry_member", name="银行")
quant.load_financial_manifest().head()

In [42]:
# ==== B21 · 组合：指数成分 → 股票池 → 行情 → 标签 ====
# 用某个月末的成分快照当股票池，是规避幸存者偏差的做法：池子里只放当时真实
# 纳入指数的股票。快照是月末的，实际研究里按月滚动取。
pool_snap = quant.load_index_weight(INDEX, start="2024-01-31", end="2024-01-31")
pool = pool_snap.stock_code.tolist()
pool_px = quant.load_panel(
    start="2024-02-01", end="2024-02-29", stocks=pool,
    fields=["close"], adjust="hfq", drop_suspended=True,
)
pool_label = quant.load_label(
    start="2024-02-01", end="2024-02-29", horizons=[5], price="close", stocks=pool,
)
pool_paired = pool_px.merge(pool_label, on=["stock_code", "trade_date"], how="left")
pool_paired.head()


,stock_code,trade_date,close,fwd_ret_close_5
0,SH600000,2024-02-01,105.63800,0.023529
1,SH600009,2024-02-01,102.38714,0.058788
2,SH600010,2024-02-01,11.92800,0.070423
3,SH600011,2024-02-01,42.35661,0.029446
4,SH600015,2024-02-01,32.94792,-0.009950


In [46]:
# ==== B22 · 组合：算因子 + 单日截面 IC ====
# 20 日动量 → 截面 z-score → 与未来 5 日收益对齐算 Spearman 相关。
# 因子只能用 t 日及以前的信息，标签在 t 日之后，方向别弄反。
history = quant.load_panel_wide("close", start="2023-12-01", end="2024-01-31", adjust="hfq")
mom20 = history / history.shift(20) - 1
mom20_z = mom20.sub(mom20.mean(axis=1), axis=0).div(mom20.std(axis=1), axis=0)
day = mom20_z.index[-1]
factor = mom20_z.loc[day].rename("mom20_z")
target = (
    quant.load_label(start=str(day.date()), end=str(day.date()), horizons=[5], price="close")
    .set_index("stock_code")["fwd_ret_close_5"]
)
single_day = pd.concat([factor, target], axis=1).dropna()
pd.DataFrame({
    "交易日": [day.date()],
    "样本数": [len(single_day)],
    "RankIC": [round(single_day.mom20_z.corr(single_day.fwd_ret_close_5, method="spearman"), 4)],
})

,交易日,样本数,RankIC
0,2024-01-31,5326,0.3445


In [47]:
# ==== B23 · 组合：IC 序列与 ICIR ====
# 逐日算截面 Rank IC，再看均值、波动、ICIR（均值 / 标准差）与胜率。
# 换别的因子时，只要把 ic_mom20 那一行替换掉即可。
ic_close = quant.load_panel_wide("close", start=START, end=END, adjust="hfq")
ic_mom20 = ic_close / ic_close.shift(20) - 1
ic_fwd5 = quant.load_label_wide(5, start=START, end=END, price="close")
ic_joined = pd.concat(
    [ic_mom20.stack().rename("factor"), ic_fwd5.stack().rename("fwd")], axis=1
).dropna()
ic_series = ic_joined.groupby(level=0).apply(
    lambda g: g["factor"].corr(g["fwd"], method="spearman")
)
pd.DataFrame({
    "交易日数": [len(ic_series)],
    "IC 均值": [round(ic_series.mean(), 4)],
    "IC 标准差": [round(ic_series.std(), 4)],
    "ICIR": [round(ic_series.mean() / ic_series.std(), 4)],
    "IC > 0 占比": [round((ic_series > 0).mean(), 4)],
})

,交易日数,IC 均值,IC 标准差,ICIR,IC > 0 占比
0,222,-0.0818,0.1985,-0.412,0.2793


In [48]:
# ==== B24 · 组合：可交易性过滤 ====
# 回测里「有行情」不等于「成交得了」：涨跌停、一字板都别当成有效成交。
# can_buy_close / can_sell_close 是清洗阶段给的保守口径，直接拿来过滤最省事。
trade = quant.load_panel(
    start="2024-01-02", end="2024-01-31",
    fields=["close", "is_st", "is_limit_up", "is_limit_down", "is_one_word_board",
            "can_buy_close", "can_sell_close", "status"],
    adjust="hfq", drop_suspended=True,
)
pd.DataFrame({
    "交易日数": [trade.trade_date.nunique()],
    "样本行数": [len(trade)],
    "ST 行": [int(trade.is_st.sum())],
    "涨停行": [int(trade.is_limit_up.sum())],
    "跌停行": [int(trade.is_limit_down.sum())],
    "一字板行": [int(trade.is_one_word_board.sum())],
    "次日可买行": [int(trade.can_buy_close.sum())],
    "次日可卖行": [int(trade.can_sell_close.sum())],
})

,交易日数,样本行数,ST 行,涨停行,跌停行,一字板行,次日可买行,次日可卖行
0,22,117381,2469,1039,749,117,116342,116632


In [49]:
# ==== B25 · 组合：个股 vs 指数累计净值 ====
# 个股用后复权 ret 复利成净值，指数用收盘点位算收益，两条曲线对齐后再看超额。
nav_stock = quant.load_panel(start=START, end=END, stocks=[STOCK], fields=["ret"], adjust="hfq")
nav_index = quant.load_index_daily(INDEX, start=START, end=END)
nav_ret = pd.concat(
    [
        nav_stock.set_index("trade_date")["ret"].rename(STOCK),
        nav_index.set_index("trade_date")["close"].pct_change().rename(INDEX),
    ],
    axis=1,
).dropna()
nav = (1 + nav_ret).cumprod()
nav["超额"] = nav[STOCK] / nav[INDEX] - 1
nav.iloc[[0, -1]]

,SH600000,000300.SH,超额
trade_date,,,
2024-01-03,1.006061,0.997621,0.008459
2024-12-31,1.616305,1.161991,0.390979


In [3]:
import quant
df = quant.load_factor_panel("2024-01-01", "2024-12-31", names=["MOM_20", "EP_TTM"])
wide = quant.load_factor_wide("MOM_20", start="2024-01-01")
df

,stock_code,trade_date,MOM_20,EP_TTM
0,BJ920000,2024-01-02,0.337423,0.053429
1,BJ920001,2024-01-02,1.081321,0.017681
2,BJ920006,2024-01-02,0.603339,0.029080
3,BJ920010,2024-01-02,0.396825,0.035705
4,BJ920014,2024-01-02,0.352614,0.039301
...,...,...,...,...
1296790,SZ301626,2024-12-31,-0.123336,0.017215
1296791,SZ301628,2024-12-31,-0.034434,0.016906
1296792,SZ301631,2024-12-31,-0.012290,0.030969
1296793,SZ301633,2024-12-31,-0.104484,0.024501


In [4]:
wide

stock_code,BJ920000,BJ920001,BJ920002,BJ920003,BJ920005,BJ920006,BJ920007,BJ920008,BJ920009,BJ920010,...,SZ301686,SZ301687,SZ301688,SZ301689,SZ301696,SZ301697,SZ301699,SZ301707,SZ301717,SZ302132
trade_date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,0.337423,1.081321,NaN,NaN,NaN,0.603339,NaN,NaN,NaN,0.396825,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.004493
2024-01-03,0.135881,0.762887,NaN,NaN,NaN,0.414074,NaN,NaN,NaN,0.322917,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009982
2024-01-04,0.169750,0.839390,NaN,NaN,NaN,0.489842,NaN,NaN,NaN,0.267465,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.040276
2024-01-05,0.089440,0.543478,NaN,NaN,NaN,0.511983,NaN,NaN,NaN,0.177291,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.069147
2024-01-08,0.039572,0.360000,NaN,NaN,NaN,0.368794,NaN,NaN,NaN,-0.021277,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.141580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-21,0.023599,0.089661,-0.030769,0.036208,0.086084,0.265060,0.023890,0.215885,-0.004755,0.070574,...,NaN,0.129174,NaN,NaN,0.003257,NaN,NaN,-0.054068,-0.076181,0.064727
2026-09-22,0.051939,0.094804,-0.050941,0.016319,0.085512,0.224274,-0.000606,0.165532,-0.034047,0.052817,...,NaN,0.080671,NaN,NaN,-0.021008,NaN,NaN,-0.067535,-0.038558,0.069009
2026-09-23,0.051508,0.102683,-0.025501,0.160131,0.065646,0.153541,-0.005993,0.124542,-0.002598,0.030769,...,NaN,0.116667,NaN,NaN,-0.011617,NaN,NaN,-0.059037,-0.069307,0.030692


当前的因子评估系统已经实现了雏形，计算包、前端以及后端都写好了demo，但是还没有真正联调验证过。我现在有几个关于已实现部分的问题，以及下一步实现的看法：
- 计算包是否实现了所需的全部数据分析功能
- 前后端应该怎么运行起来进行调试
- 如果因子评估系统已经没问题了，则下一步是：
    - 回测系统：有很多开源的回测框架可以参考，挑选一个轻量、功能齐全，且可以适配我们的数据的
    - 策略编写：需要设计一个策略设计、存储的系统
    - ML建模：这一步其实我没有理解，ML的作用是什么？这里得到的模型是根据因子值来生成买卖信号的吗？这里应该要保证速度，所以模型要使用C++来写？
    
两个关于已实现部分的问题可以现在就出方案来给我答复，关于下一步实现那部分我们可以讨论一下

- 得补一个IC 和 ICIR的cumsum
- IC分布缺横坐标